# STEP 0: 6개 공공데이터 API 통합 테스트

각 API를 최소 1회씩 호출하여 정상 응답 여부를 확인한다.
실행 전 프로젝트 루트의 `.env`에 `DATA_GO_KR_SERVICE_KEY`를 채워둘 것.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.api_visitors import get_metro_visitors, get_local_visitors
from src.api_diversity import get_tourist_diversity, get_expenditure_diversity, get_international_diversity
from src.api_resource_demand import get_tourism_service_demand, get_cultural_resource_demand
from src.api_demand_intensity import get_mobility_intensity, get_expenditure_intensity
from src.api_tourapi import get_area_based_list
from src.api_holiday import get_holiday_info, get_national_holiday_info, get_holidays_range

## 1. 방문자수 — 하루(20240701) 조회

관광객수(touNum) 등 필드가 정상적으로 채워지는지 확인.

In [ ]:
result_day = get_metro_visitors(start_ymd="20240701", end_ymd="20240701")
result_day

**결론**: (실행 후 기록) ✅ 정상 작동 / ❌ 데이터 없음 또는 에러

## 2. 방문자수 — 날짜범위(20240701~20240705) 조회

**핵심 확인사항**: totalCount가 1번(하루치)보다 늘어나는지 확인.

In [ ]:
result_range = get_metro_visitors(start_ymd="20240701", end_ymd="20240705")
result_range

In [ ]:
count_day = result_day.get("response", {}).get("body", {}).get("totalCount")
count_range = result_range.get("response", {}).get("body", {}).get("totalCount")
print(f"하루치 totalCount: {count_day}")
print(f"5일치 totalCount: {count_range}")
print(f"5일치가 더 큰가: {count_range is not None and count_day is not None and count_range > count_day}")

**결론**: (실행 후 기록) ✅ 정상 작동 (범위 확장 시 totalCount 증가 확인) / ❌ 데이터 없음 또는 에러

## 3. 관광 다양성 — 2024년 1월(baseYm=202401) 조회

**핵심 확인사항**: 2024년 데이터 존재 여부.

(area_cd는 임의의 광역 코드로 테스트, 실제 값은 지역 코드표 확인 후 조정)

In [ ]:
diversity_result = get_tourist_diversity(base_ym="202401", area_cd="11")
diversity_result

**결론**: (실행 후 기록) ✅ 정상 작동 (2024년 데이터 존재) / ❌ 데이터 없음 또는 에러

## 4. 관광 자원 수요 — 2024년 1월(baseYm=202401) 조회

**핵심 확인사항**: 2024년 데이터 존재 여부.

In [ ]:
resource_demand_result = get_tourism_service_demand(base_ym="202401", area_cd="11")
resource_demand_result

**결론**: (실행 후 기록) ✅ 정상 작동 (2024년 데이터 존재) / ❌ 데이터 없음 또는 에러

## 5. 관광 수요 강도 — 2024년 1월(baseYm=202401) 조회

**핵심 확인사항**: 2024년 데이터 존재 여부.

In [ ]:
demand_intensity_result = get_mobility_intensity(base_ym="202401", area_cd="11")
demand_intensity_result

**결론**: (실행 후 기록) ✅ 정상 작동 (2024년 데이터 존재) / ❌ 데이터 없음 또는 에러

## 5-1. (참고) TourAPI — 지역기반 관광정보 조회

요청 스펙 확정 필수 항목은 아니지만, 지역 목록 조회가 정상 동작하는지 함께 확인.

In [ ]:
tourapi_result = get_area_based_list(l_dong_regn_cd="11", content_type_id="12")
tourapi_result

**결론**: (실행 후 기록) ✅ 정상 작동 / ❌ 데이터 없음 또는 에러

## 6. 특일정보 — 2024년 5월(solYear=2024, solMonth=05) 조회

어린이날 등 5월 공휴일이 정상 조회되는지 확인.

In [ ]:
holiday_result = get_holiday_info(sol_year="2024", sol_month="05")
holiday_result

**결론**: (실행 후 기록) ✅ 정상 작동 (5월 공휴일 정상 조회) / ❌ 데이터 없음 또는 에러

## 참고: 2024~2025년 24개월치 공휴일 일괄 수집 테스트

In [ ]:
holidays_2024_2025 = get_holidays_range(start_year=2024, start_month=1, end_year=2025, end_month=12)
len(holidays_2024_2025), holidays_2024_2025[:3]

## 종합 결론

- 위 각 항목의 ✅/❌ 결과를 실행 후 이 셀에 정리할 것.
- **특히 3, 4, 5번(관광 다양성/자원 수요/수요 강도)은 2024년 데이터가 없을 경우**, 이 프로젝트에서 해당 API의 활용 범위(분석 시작 연도, 대체 데이터 소스 등)를 재조정해야 한다.
- 해당 재조정 필요 여부와 결과는 `docs/decisions_pending.md`에도 반영할 것.